# CLARE 再現実装 — 最終版
**実行前に**: `ランタイム → ランタイムのタイプを変更 → T4 GPU`

セルを **上から順に** 実行してください。

In [ ]:
# ═══════════════════════════════════════════════
# Cell 1: パッケージ & Drive マウント
# ═══════════════════════════════════════════════
!pip install -q lightgbm xgboost
!apt-get install -y -q unar

import os
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive はマウント済みです')

import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 2: RAR 展開 (初回のみ)
# ═══════════════════════════════════════════════
UNZIPPED = '/content/drive/MyDrive/doi-10.5683-sp3-h0aelt (Unzipped Files)'
WORK_DIR = '/content/CLARE'

if not os.path.isdir(f'{WORK_DIR}/ECG'):
    os.makedirs(WORK_DIR, exist_ok=True)
    for mod in ['ECG', 'EDA', 'EEG', 'Gaze', 'Labels']:
        rar = f'{UNZIPPED}/{mod}.rar'
        if os.path.exists(rar):
            print(f'{mod} 展開中...')
            os.system(f'unar -o "{WORK_DIR}" "{rar}"')
            print(f'  完了')
        else:
            print(f'  ⚠ {mod}.rar が見つかりません')
else:
    print('データはすでに展開済みです')

print('\n展開結果:')
for mod in ['ECG', 'EDA', 'EEG', 'Gaze', 'Labels']:
    exists = os.path.isdir(f'{WORK_DIR}/{mod}') or os.path.isfile(f'{WORK_DIR}/{mod}')
    print(f'  {"✓" if exists else "✗"} {mod}')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 3: 全インポート & グローバル定数
# ═══════════════════════════════════════════════
import math, copy, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt, welch, find_peaks, iirnotch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import xgboost as xgb
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
warnings.filterwarnings('ignore')

# ─── 定数 ─────────────────────────────────────
FS         = {'ECG': 512, 'EDA': 128, 'EEG': 256, 'Gaze': 50}
SEG_SEC    = 10
LABEL_TH   = 5
MODALITIES = ['ECG', 'EDA', 'EEG', 'Gaze']
DATA_ROOT  = '/content/CLARE'
SAVE_DIR   = '/content/drive/MyDrive/CLARE_models'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(SAVE_DIR, exist_ok=True)

FILE_TMPL = {
    'ECG':  lambda i: f'ecg_data_experiment_{i}.csv',
    'EDA':  lambda i: f'eda_data_experiment_{i}.csv',
    'EEG':  lambda i: f'eeg_data_exp_{i}.csv',
    'Gaze': lambda i: f'gaze_data_experiment_{i}.csv',
}
MOD_COLS = {
    'ECG':  ['ECG LL-RA CAL', 'ECG LA-RA CAL', 'ECG Vx-RL CAL'],
    'EDA':  ['GSR Conductance CAL'],
    'EEG':  ['TP9', 'AF7', 'AF8', 'TP10'],
    'Gaze': ['ET_PupilLeft', 'ET_PupilRight', 'ET_GazeLeftx', 'ET_GazeLefty'],
}
print('Device:', DEVICE)
print('定数定義 完了')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 4: 前処理関数
# ═══════════════════════════════════════════════
def preprocess_ecg(sig, fs=512):
    nyq = fs / 2
    sos = butter(4, [5/nyq, 15/nyq], btype='band', output='sos')
    out = np.zeros_like(sig, dtype=np.float32)
    for ch in range(sig.shape[1]):
        x = sosfiltfilt(sos, sig[:, ch])
        out[:, ch] = (x - x.mean()) / (x.std() + 1e-8)
    return out

def preprocess_eda_decomp(sig, fs=128):
    x = sig.ravel().astype(np.float32)
    nyq = fs / 2
    lp  = sosfiltfilt(butter(4, 3/nyq, btype='low', output='sos'), x)
    ph  = sosfiltfilt(butter(4, 0.05/nyq, btype='high', output='sos'), lp)
    return {'tonic': (lp - ph).astype(np.float32), 'phasic': ph.astype(np.float32)}

def preprocess_eeg(sig, fs=256):
    nyq = fs / 2
    bp  = butter(4, [0.4/nyq, min(128, nyq-1)/nyq], btype='band', output='sos')
    b_n, a_n = iirnotch(60/nyq, Q=30)
    nsos = np.array([[b_n[0], b_n[1], b_n[2], 1., a_n[1], a_n[2]]])
    out  = np.zeros_like(sig, dtype=np.float32)
    for ch in range(sig.shape[1]):
        out[:, ch] = sosfiltfilt(nsos, sosfiltfilt(bp, sig[:, ch]))
    return out

def preprocess_gaze(sig):
    out = np.empty_like(sig, dtype=np.float32)
    for ch in range(sig.shape[1]):
        s = pd.Series(sig[:, ch].astype(float))
        s[s == 0] = np.nan
        s = s.interpolate('linear', limit_direction='both').ffill().bfill()
        a = s.to_numpy(np.float32)
        out[:, ch] = (a - a.mean()) / (a.std() + 1e-8)
    return out

PREP_FN = {
    'ECG':  lambda s: preprocess_ecg(s, FS['ECG']),
    'EDA':  lambda s: s.astype(np.float32),
    'EEG':  lambda s: preprocess_eeg(s, FS['EEG']),
    'Gaze': lambda s: preprocess_gaze(s),
}
print('前処理関数 定義完了')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 5: 特徴抽出関数
# ═══════════════════════════════════════════════
BANDS = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,64)}

def ecg_features(seg, fs=512):
    x = seg[:, 0]
    peaks, _ = find_peaks(x, distance=int(0.3*fs))
    rr = np.diff(peaks)/fs if len(peaks) >= 2 else np.array([])
    rr = rr[(rr > 0.3) & (rr < 2.0)] if len(rr) else rr
    if len(rr) >= 2:
        hr=float((60/rr).mean()); sdn=float(rr.std())
        rms=float(np.sqrt(np.mean(np.diff(rr)**2))); p50=float(100*(np.abs(np.diff(rr))>0.05).mean())
    else:
        hr=sdn=rms=p50=0.
    bands=[0.]*4
    if len(rr) >= 4:
        t=np.cumsum(rr)-rr[0]; tu=np.arange(0, t[-1], 0.25)
        if len(tu) >= 8:
            fi,ps=welch(np.interp(tu,t,rr), fs=4., nperseg=min(len(tu),256))
            for i,(lo,hi) in enumerate([(0,.003),(.003,.04),(.04,.15),(.15,.40)]):
                m=(fi>=lo)&(fi<hi); bands[i]=float(np.trapz(ps[m],fi[m])) if m.any() else 0.
    stats=[v for c in range(seg.shape[1]) for v in [seg[:,c].mean(),seg[:,c].std(),seg[:,c].min(),seg[:,c].max()]]
    return np.nan_to_num(np.array([hr,sdn,rms,p50]+bands+stats, dtype=np.float32))

def eda_features(seg, fs=128):
    d=preprocess_eda_decomp(seg.ravel(), fs)
    t,p=d['tonic'],d['phasic']
    sl=float(np.polyfit(np.arange(len(t)),t,1)[0]) if len(t)>1 else 0.
    scl=[t.mean(),t.std(),t.min(),t.max(),sl]
    pks,pr=find_peaks(p, height=0.01, distance=fs)
    if len(pks)>0:
        scr=[p.mean(),p.std(),p.min(),p.max(),float(len(pks)),float(pr['peak_heights'].mean()),0.]
    else:
        scr=[p.mean(),p.std(),p.min(),p.max(),0.,0.,0.]
    return np.nan_to_num(np.array(scl+scr, dtype=np.float32))

def eeg_features(seg, fs=256):
    feats=[]
    for ch in range(seg.shape[1]):
        x=seg[:,ch]; d1=np.diff(x); d2=np.diff(d1)
        act=float(np.var(x)); mob=float(np.sqrt(np.var(d1)/(act+1e-12)))
        comp=float(np.sqrt(np.var(d2)/(np.var(d1)+1e-12))/(mob+1e-12))
        fi,ps=welch(x, fs=fs, nperseg=min(len(x),256))
        pn=ps/(ps.sum()+1e-12); se=float(-np.sum(pn*np.log2(pn+1e-12)))
        bp=[float(np.trapz(ps[(fi>=lo)&(fi<hi)],fi[(fi>=lo)&(fi<hi)])) if ((fi>=lo)&(fi<hi)).any() else 0.
            for lo,hi in BANDS.values()]
        feats+=[x.mean(),x.std(),act,mob,comp,se]+bp
    return np.nan_to_num(np.array(feats, dtype=np.float32))

def gaze_features(seg, fs=50):
    pupil=seg[:,0]
    sl=float(np.polyfit(np.arange(len(pupil)),pupil,1)[0]) if len(pupil)>1 else 0.
    pf=[pupil.mean(),pupil.std(),pupil.min(),pupil.max(),sl]
    blink=pupil<(pupil.mean()-2*pupil.std())
    bd,ib,s0=[],False,0
    for i,b in enumerate(blink):
        if b and not ib: ib,s0=True,i
        elif not b and ib: ib=False; bd.append((i-s0)/fs)
    bf=[float(len(bd)), float(np.mean(bd)) if bd else 0.]
    vf=[0.]*5
    if seg.shape[1] >= 4:
        vx,vy=seg[:,2],seg[:,3]
        vel=np.sqrt(np.diff(vx)**2+np.diff(vy)**2)*fs
        sac_runs=[]
        in_s,s0=False,0
        for i,v in enumerate(vel>100.):
            if v and not in_s: in_s,s0=True,i
            elif not v and in_s: in_s=False; sac_runs.append((s0,i))
        fx_durs=[(len(vel)-len(sac_runs))/fs] if sac_runs else [len(vel)/fs]
        sac_vels=[vel[s:e].mean() for s,e in sac_runs if e>s]
        vf=[float(len(fx_durs)),float(np.mean(fx_durs)) if fx_durs else 0.,
            float(len(sac_runs)),float(np.mean(sac_vels)) if sac_vels else 0.,0.]
    return np.nan_to_num(np.array(pf+bf+vf, dtype=np.float32))

FEAT_FN = {'ECG':ecg_features,'EDA':eda_features,'EEG':eeg_features,'Gaze':gaze_features}
print('特徴抽出関数 定義完了')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 6: データローダー
# ═══════════════════════════════════════════════
def _load_sig(pid_dir, mod, sess):
    path = Path(pid_dir) / FILE_TMPL[mod](sess)
    if not path.exists() or path.stat().st_size < 10:
        return None
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    cols = [c for c in MOD_COLS[mod] if c in df.columns]
    if not cols:
        return None
    arr = (df[cols]
           .interpolate('linear', limit_direction='both')
           .ffill().bfill()
           .to_numpy(np.float32))
    mask = ~np.isnan(arr).any(axis=1)
    return arr[mask] if mask.sum() >= FS[mod] * SEG_SEC else None

def _segment(sig, fs):
    seg_len = fs * SEG_SEC
    n = len(sig) // seg_len
    return sig[:n * seg_len].reshape(n, seg_len, sig.shape[1])

def load_dataset(data_root=DATA_ROOT, modalities=MODALITIES):
    root = Path(data_root)
    pids = sorted(p.name for p in (root / 'ECG').iterdir() if p.is_dir())
    print(f'参加者: {len(pids)} 名')

    X_buf  = {m: [] for m in modalities}
    y_buf, s_buf = [], []

    for subj_idx, pid in enumerate(tqdm(pids, desc='Loading')):
        label_path = root / 'Labels' / f'{pid}.csv'
        if not label_path.exists():
            continue
        try:
            labels_df = pd.read_csv(label_path)
        except Exception:
            continue

        for sess in range(4):
            col = f'level_{sess}'
            if col not in labels_df.columns:
                continue
            labels_raw = labels_df[col].dropna().to_numpy(np.float32)
            if len(labels_raw) == 0:
                continue

            segs, counts, ok = {}, [], True
            for mod in modalities:
                sig = _load_sig(root / mod / pid, mod, sess)
                if sig is None:
                    ok = False; break
                s = _segment(sig, FS[mod])
                if len(s) == 0:
                    ok = False; break
                segs[mod] = s
                counts.append(len(s))
            if not ok:
                continue

            n = min(counts + [len(labels_raw)])
            if n == 0:
                continue

            for mod in modalities:
                X_buf[mod].append(segs[mod][:n])
            y_buf.append((labels_raw[:n] >= LABEL_TH).astype(np.int64))
            s_buf.append(np.full(n, subj_idx, np.int64))

    if not y_buf:
        raise ValueError('データを1件も読めませんでした')

    X    = {m: np.concatenate(X_buf[m], axis=0) for m in modalities}
    y    = np.concatenate(y_buf)
    subj = np.concatenate(s_buf)
    print(f'セグメント数: {len(y)}  (低負荷: {(y==0).sum()}, 高負荷: {(y==1).sum()})')
    return X, y, subj

print('データローダー定義完了')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 7: データ読み込み実行
# ═══════════════════════════════════════════════
X_raw, y, subject_ids = load_dataset()

In [ ]:
# ═══════════════════════════════════════════════
# Cell 8: 特徴行列を構築（古典的ML用）
# ═══════════════════════════════════════════════
def build_features(X_raw, modalities=MODALITIES):
    blocks = []
    for mod in modalities:
        segs = X_raw[mod]
        print(f'  {mod}: {len(segs)} segments ...')
        prep = np.stack([PREP_FN[mod](segs[i]) for i in tqdm(range(len(segs)), leave=False)])
        feat = np.stack([FEAT_FN[mod](prep[i]) for i in range(len(prep))])
        print(f'    → {feat.shape[1]} features')
        blocks.append(feat)
    return np.concatenate(blocks, axis=1)

print('特徴量を構築中...')
X_feat = build_features(X_raw)
print(f'完了: {X_feat.shape}')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 9: 古典的ML — 8モデル評価
# ═══════════════════════════════════════════════
def get_ml_models():
    S = StandardScaler
    return {
        'GB':      Pipeline([('s',S()),('c',GradientBoostingClassifier(n_estimators=200,max_depth=4,random_state=42))]),
        'LGBM':    Pipeline([('s',S()),('c',lgb.LGBMClassifier(n_estimators=200,random_state=42,verbose=-1))]),
        'LDA':     Pipeline([('s',S()),('c',LinearDiscriminantAnalysis())]),
        'LR':      Pipeline([('s',S()),('c',LogisticRegression(max_iter=1000,random_state=42))]),
        'MLP':     Pipeline([('s',S()),('c',MLPClassifier(hidden_layer_sizes=(256,128),max_iter=300,random_state=42))]),
        'RF':      Pipeline([('s',S()),('c',RandomForestClassifier(n_estimators=200,random_state=42))]),
        'SVM':     Pipeline([('s',S()),('c',SVC(kernel='rbf',C=1.,random_state=42))]),
        'XGBoost': Pipeline([('s',S()),('c',xgb.XGBClassifier(n_estimators=200,eval_metric='logloss',random_state=42,verbosity=0))]),
    }

def kfold_eval(mdl, X, y, k=10):
    accs, f1s = [], []
    for tr, te in StratifiedKFold(k, shuffle=True, random_state=42).split(X, y):
        m = copy.deepcopy(mdl); m.fit(X[tr], y[tr]); p = m.predict(X[te])
        accs.append(accuracy_score(y[te], p))
        f1s.append(f1_score(y[te], p, average='weighted', zero_division=0))
    return np.mean(accs), np.mean(f1s)

def loso_eval(mdl, X, y, subj):
    accs, f1s = [], []
    for s in np.unique(subj):
        te = subj == s; tr = ~te
        if tr.sum() == 0 or te.sum() == 0: continue
        m = copy.deepcopy(mdl); m.fit(X[tr], y[tr]); p = m.predict(X[te])
        accs.append(accuracy_score(y[te], p))
        f1s.append(f1_score(y[te], p, average='weighted', zero_division=0))
    return np.mean(accs), np.mean(f1s)

ml_results = {}
for name, mdl in tqdm(get_ml_models().items(), desc='ML Models'):
    a10, f10 = kfold_eval(mdl, X_feat, y)
    alo, flo = loso_eval(mdl, X_feat, y, subject_ids)
    ml_results[name] = {'10-fold': (a10, f10), 'LOSO': (alo, flo)}
    print(f'{name:<8} 10-fold: acc={a10*100:.2f}% f1={f10*100:.2f}% | LOSO: acc={alo*100:.2f}% f1={flo*100:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 10: DLモデル定義
# ═══════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, a=4., g=2.): super().__init__(); self.a, self.g = a, g
    def forward(self, lo, tg):
        ce = F.cross_entropy(lo, tg, reduction='none')
        return (self.a * (1 - torch.exp(-ce)) ** self.g * ce).mean()

class ConvBlock(nn.Module):
    def __init__(self, ic, oc, k=3):
        super().__init__()
        p = k // 2
        self.b = nn.Sequential(
            nn.Conv1d(ic,oc,k,padding=p), nn.BatchNorm1d(oc), nn.ReLU(),
            nn.Conv1d(oc,oc,k,padding=p), nn.BatchNorm1d(oc), nn.ReLU(),
            nn.MaxPool1d(2))
    def forward(self, x): return self.b(x)

class Branch(nn.Module):
    def __init__(self, ic, em=128):
        super().__init__()
        ch = [32, 64, 128, 256]
        self.cnn = nn.Sequential(
            ConvBlock(ic,ch[0]), ConvBlock(ch[0],ch[1]),
            ConvBlock(ch[1],ch[2]), ConvBlock(ch[2],ch[3]))
        self.pool = nn.AdaptiveAvgPool1d(1); self.fc = nn.Linear(ch[-1], em)
    def forward(self, x): return self.fc(self.pool(self.cnn(x)).squeeze(-1))

class MultiModalCNN(nn.Module):
    def __init__(self, mod_channels, em=128, nc=2):
        super().__init__()
        self.ns = list(mod_channels.keys())
        self.br = nn.ModuleDict({n: Branch(c, em) for n, c in mod_channels.items()})
        self.hd = nn.Sequential(nn.Linear(em*len(mod_channels),256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256,nc))
    def forward(self, xd): return self.hd(torch.cat([self.br[n](xd[n]) for n in self.ns], dim=-1))

class PosEnc(nn.Module):
    def __init__(self, d, ml=5000, dr=0.1):
        super().__init__(); self.drop = nn.Dropout(dr)
        pe = torch.zeros(ml, d)
        pos = torch.arange(ml).unsqueeze(1).float()
        div = torch.exp(torch.arange(0,d,2).float() * (-math.log(10000.)/d))
        pe[:,0::2] = torch.sin(pos*div); pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return self.drop(x + self.pe[:, :x.size(1)])

class TBlock(nn.Module):
    def __init__(self, d, h, f, dr=0.1):
        super().__init__()
        self.at = nn.MultiheadAttention(d, h, dropout=dr, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d,f), nn.GELU(), nn.Dropout(dr), nn.Linear(f,d))
        self.n1 = nn.LayerNorm(d); self.n2 = nn.LayerNorm(d); self.dr = nn.Dropout(dr)
    def forward(self, x):
        a, _ = self.at(x,x,x); x = self.n1(x + self.dr(a))
        return self.n2(x + self.dr(self.ff(x)))

class MultiModalTransformer(nn.Module):
    def __init__(self, mod_channels, sl=128, d=128, h=4, nb=4, f=256, dr=0.1, nc=2):
        super().__init__()
        self.ns = list(mod_channels.keys()); self.sl = sl
        self.pr = nn.Linear(sum(mod_channels.values()), d)
        self.pe = PosEnc(d, max_len=sl+1, dr=dr)
        self.bl = nn.ModuleList([TBlock(d,h,f,dr) for _ in range(nb)])
        self.hd = nn.Sequential(
            nn.Linear(d,256), nn.ReLU(), nn.Dropout(dr),
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(dr), nn.Linear(128,nc))
    def _rs(self, x): return F.interpolate(x.float(), size=self.sl, mode='linear', align_corners=False)
    def forward(self, xd):
        h = torch.cat([self._rs(xd[n]) for n in self.ns], dim=1).permute(0,2,1)
        h = self.pe(self.pr(h))
        for b in self.bl: h = b(h)
        return self.hd(h.mean(1))

print('DLモデル定義完了')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 11: DL用前処理 & 学習ユーティリティ
# ═══════════════════════════════════════════════
def prep_dl(X_raw, modalities=MODALITIES):
    res = {}
    for mod in modalities:
        segs = X_raw[mod]  # (N, T, C)
        p = np.stack([PREP_FN[mod](segs[i]) for i in range(len(segs))])
        if p.ndim == 2: p = p[:, :, None]
        res[mod] = torch.from_numpy(p.transpose(0, 2, 1).astype(np.float32))  # (N, C, T)
    return res

def make_ds(Xt, y, mods):
    return TensorDataset(*[Xt[m] for m in mods], torch.from_numpy(y).long())

def run_fold(model_cls, kwargs, Xt, y, mods, tr, te,
             ep=100, bs=256, mtype='cnn', save_path=None):
    ds  = make_ds(Xt, y, mods)
    tdl = DataLoader(Subset(ds, tr), batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True)
    vdl = DataLoader(Subset(ds, te), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)

    mdl  = model_cls(**kwargs).to(DEVICE)
    crit = FocalLoss(4., 2.)
    opt  = (torch.optim.Adadelta(mdl.parameters(), lr=5e-3, rho=.95)
            if mtype == 'cnn'
            else torch.optim.Adam(mdl.parameters(), lr=1e-4))

    best_acc, best_state = 0., None
    for ep_i in range(ep):
        mdl.train()
        for batch in tdl:
            *xs, lb = batch
            xd = {m: xs[i].to(DEVICE) for i, m in enumerate(mods)}
            opt.zero_grad(); crit(mdl(xd), lb.to(DEVICE)).backward(); opt.step()
        # val accuracy をトラッキングしてベストを保存
        mdl.eval(); ps, ts = [], []
        with torch.no_grad():
            for batch in vdl:
                *xs, lb = batch
                xd = {m: xs[i].to(DEVICE) for i, m in enumerate(mods)}
                ps.extend(mdl(xd).argmax(-1).cpu().numpy()); ts.extend(lb.numpy())
        acc = accuracy_score(ts, ps)
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.cpu().clone() for k, v in mdl.state_dict().items()}

    mdl.load_state_dict(best_state)

    if save_path:
        torch.save({'model_state_dict': best_state,
                    'model_class': model_cls.__name__,
                    'kwargs': kwargs, 'mods': mods}, save_path)
        print(f'  💾 {os.path.basename(save_path)}')

    mdl.eval(); ps, ts = [], []
    with torch.no_grad():
        for batch in vdl:
            *xs, lb = batch
            xd = {m: xs[i].to(DEVICE) for i, m in enumerate(mods)}
            ps.extend(mdl(xd).argmax(-1).cpu().numpy()); ts.extend(lb.numpy())
    return accuracy_score(ts, ps), f1_score(ts, ps, average='weighted', zero_division=0)

print('DL学習ユーティリティ定義完了')
print('DL用前処理中...')
X_dl = prep_dl(X_raw)
mod_channels = {m: X_dl[m].shape[1] for m in MODALITIES}
print('完了:', {m: v.shape for m, v in X_dl.items()})

In [ ]:
# ═══════════════════════════════════════════════
# Cell 12: CNN — 10-fold CV & LOSO (+ 保存)
# ═══════════════════════════════════════════════
cnn_kw = dict(mod_channels=mod_channels, em=128, nc=2)
idx = np.arange(len(y))
cnn_a10, cnn_f10, cnn_alo, cnn_flo = [], [], [], []
best_cnn_f1, best_cnn_fold = 0., 0

print('=== CNN 10-fold CV ===')
for fold, (tr, te) in enumerate(StratifiedKFold(10, shuffle=True, random_state=42).split(idx, y)):
    a, f = run_fold(MultiModalCNN, cnn_kw, X_dl, y, MODALITIES, tr, te,
                    ep=100, mtype='cnn')
    cnn_a10.append(a); cnn_f10.append(f)
    print(f'  Fold {fold+1:2d}: acc={a*100:.2f}% f1={f*100:.2f}%')
    if f > best_cnn_f1:
        best_cnn_f1 = f; best_cnn_fold_idx = (tr, te)

print(f'  平均: acc={np.mean(cnn_a10)*100:.2f}% f1={np.mean(cnn_f10)*100:.2f}%')

# ベストfoldを保存
print('  ベストfoldを保存中...')
run_fold(MultiModalCNN, cnn_kw, X_dl, y, MODALITIES,
         best_cnn_fold_idx[0], best_cnn_fold_idx[1],
         ep=100, mtype='cnn',
         save_path=f'{SAVE_DIR}/cnn_best_fold.pt')

print('\n=== CNN LOSO ===')
best_cnn_loso_f1, best_cnn_loso_idx = 0., None
for s in np.unique(subject_ids):
    te = np.where(subject_ids == s)[0]; tr = np.where(subject_ids != s)[0]
    a, f = run_fold(MultiModalCNN, cnn_kw, X_dl, y, MODALITIES, tr, te,
                    ep=100, mtype='cnn')
    cnn_alo.append(a); cnn_flo.append(f)
    print(f'  Subj {s+1:2d}: acc={a*100:.2f}% f1={f*100:.2f}%')
    if f > best_cnn_loso_f1:
        best_cnn_loso_f1 = f; best_cnn_loso_idx = (tr, te)

print(f'  平均: acc={np.mean(cnn_alo)*100:.2f}% f1={np.mean(cnn_flo)*100:.2f}%')

# LOSOベストを保存
run_fold(MultiModalCNN, cnn_kw, X_dl, y, MODALITIES,
         best_cnn_loso_idx[0], best_cnn_loso_idx[1],
         ep=100, mtype='cnn',
         save_path=f'{SAVE_DIR}/cnn_best_loso.pt')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 13: Transformer — 10-fold CV & LOSO (+ 保存)
# ═══════════════════════════════════════════════
sl = min(v.shape[-1] for v in X_dl.values())
tr_kw = dict(mod_channels=mod_channels, sl=sl, d=128, h=4, nb=4, f=256, nc=2)
tr_a10, tr_f10, tr_alo, tr_flo = [], [], [], []
best_tr_f1, best_tr_fold_idx = 0., None

print('=== Transformer 10-fold CV ===')
for fold, (tr, te) in enumerate(StratifiedKFold(10, shuffle=True, random_state=42).split(idx, y)):
    a, f = run_fold(MultiModalTransformer, tr_kw, X_dl, y, MODALITIES, tr, te,
                    ep=100, mtype='transformer')
    tr_a10.append(a); tr_f10.append(f)
    print(f'  Fold {fold+1:2d}: acc={a*100:.2f}% f1={f*100:.2f}%')
    if f > best_tr_f1:
        best_tr_f1 = f; best_tr_fold_idx = (tr, te)

print(f'  平均: acc={np.mean(tr_a10)*100:.2f}% f1={np.mean(tr_f10)*100:.2f}%')

run_fold(MultiModalTransformer, tr_kw, X_dl, y, MODALITIES,
         best_tr_fold_idx[0], best_tr_fold_idx[1],
         ep=100, mtype='transformer',
         save_path=f'{SAVE_DIR}/transformer_best_fold.pt')

print('\n=== Transformer LOSO ===')
best_tr_loso_f1, best_tr_loso_idx = 0., None
for s in np.unique(subject_ids):
    te = np.where(subject_ids == s)[0]; tr = np.where(subject_ids != s)[0]
    a, f = run_fold(MultiModalTransformer, tr_kw, X_dl, y, MODALITIES, tr, te,
                    ep=100, mtype='transformer')
    tr_alo.append(a); tr_flo.append(f)
    print(f'  Subj {s+1:2d}: acc={a*100:.2f}% f1={f*100:.2f}%')
    if f > best_tr_loso_f1:
        best_tr_loso_f1 = f; best_tr_loso_idx = (tr, te)

print(f'  平均: acc={np.mean(tr_alo)*100:.2f}% f1={np.mean(tr_flo)*100:.2f}%')

run_fold(MultiModalTransformer, tr_kw, X_dl, y, MODALITIES,
         best_tr_loso_idx[0], best_tr_loso_idx[1],
         ep=100, mtype='transformer',
         save_path=f'{SAVE_DIR}/transformer_best_loso.pt')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 14: 結果まとめ & 論文値との比較
# ═══════════════════════════════════════════════
rows = []
for nm, r in ml_results.items():
    rows.append({'Model': nm,
                 '10-fold Acc': f"{r['10-fold'][0]*100:.2f}%",
                 '10-fold F1':  f"{r['10-fold'][1]*100:.2f}%",
                 'LOSO Acc':    f"{r['LOSO'][0]*100:.2f}%",
                 'LOSO F1':     f"{r['LOSO'][1]*100:.2f}%"})
rows.append({'Model':'CNN',
             '10-fold Acc': f"{np.mean(cnn_a10)*100:.2f}%",
             '10-fold F1':  f"{np.mean(cnn_f10)*100:.2f}%",
             'LOSO Acc':    f"{np.mean(cnn_alo)*100:.2f}%",
             'LOSO F1':     f"{np.mean(cnn_flo)*100:.2f}%"})
rows.append({'Model':'Transformer',
             '10-fold Acc': f"{np.mean(tr_a10)*100:.2f}%",
             '10-fold F1':  f"{np.mean(tr_f10)*100:.2f}%",
             'LOSO Acc':    f"{np.mean(tr_alo)*100:.2f}%",
             'LOSO F1':     f"{np.mean(tr_flo)*100:.2f}%"})

df_res = pd.DataFrame(rows).set_index('Model')
print('\n===== 再現実装 結果 =====')
print(df_res.to_string())

print('\n===== 論文記載値 (参考) =====')
paper = pd.DataFrame([
    {'Model':'Transformer (paper)','10-fold Acc':'85.58%','10-fold F1':'81.18%','LOSO Acc':'72.70%','LOSO F1':'69.46%'},
    {'Model':'CNN (paper)',        '10-fold Acc':'78.45%','10-fold F1':'76.41%','LOSO Acc':'—',    'LOSO F1':'—'},
    {'Model':'RF (paper)',         '10-fold Acc':'75.12%','10-fold F1':'72.83%','LOSO Acc':'64.18%','LOSO F1':'58.21%'},
]).set_index('Model')
print(paper.to_string())

print(f'\n保存先: {SAVE_DIR}/')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') // 1024 // 1024
    print(f'  {f}  ({size} MB)')